<a href="https://colab.research.google.com/github/alwhshalkasr267-design/FinalProject/blob/main/FinalProject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connect drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load Data

In [2]:
import os
import pandas as pd

folder_path = '/content/drive/MyDrive/Obstacles_dataset'
classes_names = os.listdir(folder_path)
print("Dataset Classes: ", classes_names)

Dataset Classes:  ['chair', 'door', 'fence', 'stairs', 'pothole', 'obstacle', 'table', 'plant', 'garbage_bin', 'vehicle']


In [3]:
i = 1
for folder in sorted(os.listdir(folder_path)):
    full_folder_path = os.path.join(folder_path, folder)
    if os.path.isdir(full_folder_path):
        num_images = len(os.listdir(full_folder_path))
        print(f"{i} Number of [{folder}] Class : {num_images} images")
        i += 1

1 Number of [chair] Class : 407 images
2 Number of [door] Class : 242 images
3 Number of [fence] Class : 179 images
4 Number of [garbage_bin] Class : 175 images
5 Number of [obstacle] Class : 423 images
6 Number of [plant] Class : 139 images
7 Number of [pothole] Class : 706 images
8 Number of [stairs] Class : 504 images
9 Number of [table] Class : 185 images
10 Number of [vehicle] Class : 604 images


In [4]:
images = []
labels = []

for label_indx, classes_name in enumerate(classes_names):
  class_dir = os.path.join(folder_path, classes_name)
  if os.path.isdir(class_dir):
        print(f"Class [{classes_name}] is done ,  It got label number: {label_indx}")
        for file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, file)
            images.append(img_path)
            labels.append(str(label_indx))
print(f"\nTotal images collected : {len(images)}")

Class [chair] is done ,  It got label number: 0
Class [door] is done ,  It got label number: 1
Class [fence] is done ,  It got label number: 2
Class [stairs] is done ,  It got label number: 3
Class [pothole] is done ,  It got label number: 4
Class [obstacle] is done ,  It got label number: 5
Class [table] is done ,  It got label number: 6
Class [plant] is done ,  It got label number: 7
Class [garbage_bin] is done ,  It got label number: 8
Class [vehicle] is done ,  It got label number: 9

Total images collected : 3564


In [5]:
df = pd.DataFrame({
    "image" : images,
    "label" : labels
})

df = df.sample(frac=1).reset_index(drop=True)  #Mixing up the raws
df.head()

,image,label
0,/content/drive/MyDrive/Obstacles_dataset/stair...,3
1,/content/drive/MyDrive/Obstacles_dataset/vehic...,9
2,/content/drive/MyDrive/Obstacles_dataset/plant...,7
3,/content/drive/MyDrive/Obstacles_dataset/chair...,0
4,/content/drive/MyDrive/Obstacles_dataset/vehic...,9


In [6]:
from PIL import Image
from tqdm import tqdm

bad = []

for p in tqdm(df["image"]):
  try:
    with Image.open(p) as im:
      im.load()
  except:
    bad.append(p)

print("Bad images : ", len(bad))

df = df[~df["image"].isin(bad)].reset_index(drop=True)
print("Remaining images : ", len(df) )

100%|██████████| 3564/3564 [21:24<00:00,  2.78it/s]

Bad images :  0
Remaining images :  3564


In [7]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train samples      : {len(train_df)}")
print(f"Validation samples : {len(val_df)}")
print(f"Test samples       : {len(test_df)}")
print(f"Total after split  : {len(train_df) + len(val_df) + len(test_df)}")

train_df.head()

Train samples      : 2138
Validation samples : 713
Test samples       : 713
Total after split  : 3564


,image,label
0,/content/drive/MyDrive/Obstacles_dataset/potho...,4
1,/content/drive/MyDrive/Obstacles_dataset/fence...,2
2,/content/drive/MyDrive/Obstacles_dataset/obsta...,5
3,/content/drive/MyDrive/Obstacles_dataset/stair...,3
4,/content/drive/MyDrive/Obstacles_dataset/vehic...,9


In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import gradio as gr
from PIL import Image
import numpy as np

try:
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes_names, yticklabels=classes_names)
    plt.show()
except NameError:
    pass

def predict_image(img):
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    try:
        prediction = model.predict(img_array)
        return classes_names[np.argmax(prediction)]
    except NameError:
        return "Error: Model not found"

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Text(label="Result"),
    title="Image Classification System"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43e00a51e375e49c3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
